In [1]:
cd /workspace/llm-graph-construction

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
!pip install -q flair==0.12.2

In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.9 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
from collections import defaultdict

In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from training.train_and_evaluate_relation_extraction import *

/workspace/llm-graph-construction


/workspace/llm-graph-construction/graph_building/llm/OpenChat.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434',


In [7]:
possible_scenarios = ["ignore_fp_against_gold", "ignore_fp_against_closure", "include_fp_against_gold"]
eval_scenario = "include_fp_against_gold"

possible_event_extractors = ["spert", "flair", "bert"]
event_extractor = "bert"

# Predict all relations

In [8]:
test_name = "i2b2"
_, _, dataset_test = load_stored_dataset_combination_graph(balanced=False, dataset=test_name)
model = torch.load("evaluation_results/bimodal-model-" + test_name + ".pt")

In [9]:
dataset_test.generated = list(filter(lambda x: x is not None, map(window_text, dataset_test.generated)))
dataLoader = DataLoader(dataset_test, batch_size=1)

In [10]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

MultiModalPrediction(
  (graph_model): GraphEncoder(
    (criterion): CrossEntropyLoss()
    (softmax): Softmax(dim=1)
    (convs): ModuleList(
      (0-1): 2 x TemporalRelationAggregation()
    )
    (lns): ModuleList(
      (0): LayerNorm((50,), eps=1e-05, elementwise_affine=True)
    )
    (linear): Linear(in_features=50, out_features=50, bias=True)
    (post_mp): Sequential(
      (0): Linear(in_features=100, out_features=50, bias=True)
      (1): Dropout(p=0.2, inplace=False)
      (2): LeakyReLU(negative_slope=0.01)
      (3): Linear(in_features=50, out_features=3, bias=True)
    )
  )
  (text_model): EntityBERTtextEncoder(
    (EntityBert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30539, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )

In [11]:
import os
relation_types = ["BEFORE", "AFTER", "OVERLAP"]
pregenerated_file = "pregenerated/raw_predictions_" + test_name + ".pt"
if os.path.isfile(pregenerated_file):
    predictions, correct_graph, texts_with_events = torch.load(pregenerated_file)
else:
    predictions = []
    correct_graph = []
    texts_with_events = []
    for graph in dataLoader:
        graph.to(device)
        prediction = model(graph, graph.y)
        # predictions = np.argmax(logits, axis=-1)
        cls = np.argmax(prediction["predictions"].cpu().detach().numpy(), axis=-1)
        for i in range(len(graph["text"])):
            event1 = (graph["event1_start"][i], graph["event1_end"][i], graph["text"][i][graph["event1_start"][i]:graph["event1_end"][i]])
            event2 = (graph["event2_start"][i], graph["event2_end"][i], graph["text"][i][graph["event2_start"][i]:graph["event2_end"][i]])
            predictions.append((event1, relation_types[cls[i]], event2))
            correct_graph.append((event1, relation_types[graph.y[i]], event2))
            texts_with_events.append((graph["text"], graph["event1_start"], graph["event1_end"], graph["event1_start"], graph["event1_end"]))
    torch.save((predictions, correct_graph, texts_with_events), pregenerated_file)

In [12]:
all_texts = set(dataLoader.dataset.df["text"])
def get_original_text(edited_text, text_points):
    new_text_points = [tp for tp in text_points]
    for i in range(len(text_points)):
        if text_points[i] >= edited_text.find("<e1>"):
            new_text_points[i] -= 4
        if text_points[i] >= edited_text.find("<e2>"):
            new_text_points[i] -= 4
        if text_points[i] >= edited_text.find("</e1>"):
            new_text_points[i] -= 5
        if text_points[i] >= edited_text.find("</e2>"):
            new_text_points[i] -= 5
    edited_text = edited_text.replace("<e1>", "").replace("<e2>", "").replace("</e1>", "").replace("</e2>", "")
    for text in all_texts:
        if edited_text in text:
            offset = text.find(edited_text)
            for i in range(len(text_points)):
                new_text_points[i] += offset
            return text, new_text_points
    print("error")
    return edited_text, new_text_points

# Get all events

In [13]:
events = set([r[0] for r in correct_graph] + [r[2] for r in correct_graph])

# Predict events using multiple systems

In [14]:
if event_extractor == "flair":
    from flair.models import SequenceTagger
    from flair.embeddings import TransformerWordEmbeddings
    from flair.models import SequenceTagger
    from flair.trainers import ModelTrainer
    from flair.data import Sentence
    def get_predictions(tagged_sentence, original_text):
        predicted_events = []
        curent_event = [0,0]
        in_event = False
        offset = 0
        for word in tagged_sentence:
            offset = original_text.find(word.text, offset)
            if word.tag == "B":
                if in_event:
                    # complete curent event
                    predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
                    current_event = [offset, offset]
                    in_event = False
                in_event = True
                curent_event[0] = offset
                curent_event[1] = offset + len(word.text)
            elif word.tag == "I" and in_event == True:
                curent_event[1] = offset + len(word.text)
            elif word.tag == "I" and in_event == False:
                # ignore events without a beginning
                # in_event = True
                # curent_event[0] = offset
                # curent_event[1] = offset + len(word.text)
                pass
            elif word.tag == "X" and in_event == True:
                # complete curent event
                predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
                curent_event = [offset, offset]
                in_event = False
            
            
            offset += len(word.text)
        return predicted_events

    if test_name == "thyme":
        model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'/final-model.pt')
    else:
        model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'2/final-model.pt')
    def extract_events(text):
        sentence = Sentence(text)
        # predict tags and print
        model.predict(sentence)
        predictions = get_predictions(sentence, text)
        return predictions

elif event_extractor == "spert":
    import nltk
    import json
    nltk.download('punkt_tab')
    def split_document_if_too_long(text):
        max_length = 700 # 700 characters
        def sentence_spans(txt):
            tokens=nltk.sent_tokenize(txt)
            offset = 0
            for token in tokens:
                offset = txt.find(token, offset)
                yield token, offset, offset+len(token)
                offset += len(token)
        sentences = sentence_spans(text)
    
        documents = []
        start_offset = 0
        end_offset = 0
        document_offsets = []
        document_token_spans = []
        for sentence in sentences:
            new_document_length = sentence[2] - start_offset
            if end_offset - start_offset > 0 and new_document_length > max_length:
                doc_text = text[start_offset: end_offset]
                documents.append({"tokens": nltk.word_tokenize(doc_text)})
                document_offsets.append(start_offset)
                document_token_spans.append(list(spans(doc_text)))
                start_offset = end_offset
            end_offset = sentence[2]
        if end_offset - start_offset > 0:
            doc_text = text[start_offset: end_offset]
            documents.append({"tokens": nltk.word_tokenize(doc_text)})
            document_offsets.append(start_offset)
            document_token_spans.append(list(spans(doc_text)))
        return documents, document_offsets, document_token_spans
    def convert_text(text):
        tokens=nltk.word_tokenize(text)
        # Serializing json
        json_object, document_offsets, document_token_spans = split_document_if_too_long(text)
         
        # Writing to sample.json
        with open("/workspace/spert/data/predict.json", "w") as outfile:
            outfile.write(json.dumps(json_object))
        return document_offsets, document_token_spans
    def prepare_config():
        with open("/workspace/spert/configs/predict.conf", "w") as outfile:
            outfile.write("[1]" + "\n")
            outfile.write("model_type = spert" + "\n")
            outfile.write("model_path = data/models/" + test_name + "\n")
            outfile.write("tokenizer_path = data/models/" + test_name + "\n")
            outfile.write("dataset_path = data/predict.json" + "\n")
            outfile.write("types_path = data/datasets/i2b2/i2b2_types.json" + "\n")
            outfile.write("predictions_path = data/predictions.json" + "\n")
            outfile.write("spacy_model = en_core_web_sm" + "\n")
            outfile.write("eval_batch_size = 1" + "\n")
            outfile.write("rel_filter_threshold = 0.4" + "\n")
            outfile.write("size_embedding = 25" + "\n")
            outfile.write("prop_drop = 0.1" + "\n")
            outfile.write("max_span_size = 10" + "\n")
            outfile.write("sampling_processes = 4" + "\n")
            outfile.write("max_pairs = 1000" + "\n")

    def spans(txt):
        tokens=nltk.word_tokenize(txt)
        offset = 0
        for token in tokens:
            offset = txt.find(token, offset)
            yield token, offset, offset+len(token)
            offset += len(token)
            
    def extract_events(text):
        document_offsets, document_token_spans = convert_text(text)
        prepare_config()
        
        bash_command = 'bash -c "cd /workspace/spert; python spert.py predict --config configs/predict.conf"'
        os.system(bash_command)

        with open('/workspace/spert/data/predictions.json', 'r') as file:
            predictions = json.load(file)
        predicted_events = []
        print(predictions)
        for i, doc_predictions in enumerate(predictions):
            for pred in doc_predictions["entities"]:
                char_start = document_token_spans[i][pred["start"]][1] + document_offsets[i]
                # print(pred["end"], len(document_token_spans[i]))
                char_end = document_token_spans[i][pred["end"]-1][2] + document_offsets[i]
                if (pred["start"] >= pred["end"]):
                    print("End ni po start:", pred["start"], pred["end"])
                # print(char_start, char_end, i)
                predicted_events.append((char_start, char_end, text[char_start:char_end]))
        return predicted_events
    pass

elif event_extractor == "bert":
    def split_text_into_sentenes(text):
        tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
        sentences = list(tokenizer.span_tokenize(text))
        return sentences
    def extract_events(text):
        event_extraction_model = torch.load("event-model.pt", map_location=torch.device('cpu'))
        sentence_idxs = split_text_into_sentenes(text)
        events = []
        for sentence_idx in sentence_idxs:
            tokenized = event_extraction_model.tokenizer(text[sentence_idx[0]:sentence_idx[1]], return_tensors="pt")
            classification = event_extraction_model(tokenized)
            predictions = torch.argmax(classification["results"], axis=1)
            start = 0
            end = 0
            inside_event = False
            for i, prediction in enumerate(predictions):
                if prediction == 1:
                    if tokenized.token_to_chars(i) is None:
                        continue
                    if not inside_event:
                        start = tokenized.token_to_chars(i).start
                    end = tokenized.token_to_chars(i).end
                    inside_event = True
                else:
                    if inside_event:
                        # end this event
                        events.append((sentence_idx[0] + start, sentence_idx[0] + end, text[sentence_idx[0] + start:sentence_idx[0] + end]))
                    inside_event = False
            if inside_event:
                # end this event
                events.append((sentence_idx[0] + start, sentence_idx[0] + end, text[sentence_idx[0] + start:sentence_idx[0] + end]))
        return events

In [15]:
import torch
from pipeline.pipeline import generate_event_pairs, construct_basic_dataframe, construct_dataset_with_graphs, construct_dataset_no_graph

relation_detection_model = torch.load("best-models/relation-detection-model.pt", map_location=device)
def recognize_relations(text, events, correct_relations=None):
    relations_between_events = []

    if eval_scenario == "ignore_fp_against_gold":
        for relation in correct_relations:
            event1 = most_simmilar_event(events, relation[0])
            event2 = most_simmilar_event(events, relation[2])
            if event1 is not None and event2 is not None:
                relations_between_events.append((event1, event2))

    if eval_scenario == "ignore_fp_against_closure" or eval_scenario == "include_fp_against_gold":
        pairs = generate_event_pairs(text, events)
        for pair in pairs:
            relations_between_events.append(pair)
        # dataframe = construct_basic_dataframe(text, pairs, 0)
        # dataset = construct_dataset_no_graph(text, dataframe, 0) # this is only ok if using the relation_detection_model
        # loader = DataLoader(dataset, batch_size=16)
        # pair_index = 0
        # for batch in loader:
        #     batch.to(device)
        #     labels = batch.y
        #     relation_detection = relation_detection_model(data=batch, labels=labels)
        #     predictions = np.argmax(relation_detection["logits"].detach().cpu(), axis=1)
        #     # print(predictions)
        #     for p in predictions:
        #         if p == 1:
        #             relations_between_events.append(pairs[pair_index])
        #         pair_index += 1
    return relations_between_events

In [15]:
extract_events("Addmission Date: 2020-03-11 Discharge Date: 2020-03-23 Service: MEDICINE History of Present illness:")

[(0, 10, 'Addmission'), (28, 37, 'Discharge')]

# Simulate pipeline

In [16]:
# simulate imperfect event extraction performance
import random
remove_percentage = 0.1
number_of_removed = int(remove_percentage * len(events))
removed_events = random.sample(events, number_of_removed)

In [ ]:
# Use events detected by event extraction model
events

In [ ]:
correct_predictions = 0
all_predictions = 0
all_correct_predictions = 0

for predicted, correct in zip(predictions, correct_graph):
    event1 = predicted[0]
    event2 = predicted[2]
    if event1 in removed_events or event2 in removed_events:
        all_correct_predictions += 1
    else:
        all_correct_predictions += 1
        all_predictions += 1
        if predicted[1] == correct[1]:
            correct_predictions += 1

In [ ]:
print("Accuracy", correct_predictions / all_predictions)
print("Accuracy global", correct_predictions / all_correct_predictions)

# Compute pipeline accuracy for event extractor

In [34]:
import nltk
cache = {}

documents = set()
document_texts = {}
gold_graphs = {}
predicted_graphs = {}
additional_relations = defaultdict(set)

events_found = set()
events_all = set()

gold_graphs_precomputed = defaultdict(list)
for predicted, correct, text_with_events in zip(predictions, correct_graph, texts_with_events):
    text_with_events = [x[0] for x in text_with_events]
    text, points = get_original_text(text_with_events[0], text_with_events[1:5])
    event1_start = points[0]
    event1_end = points[1]
    event2_start = points[2]
    event2_end = points[3]
    document = hash(text)

    gold_graphs_precomputed[document].append(((event1_start, event1_end, correct[0][2]), correct[1], (event2_start, event2_end, correct[2][2])))

for predicted, correct, texts in zip(predictions, correct_graph, texts_with_events):
    texts = [x[0] for x in texts]
    
    text, points = get_original_text(texts[0], texts[1:5])
    event1_start = points[0]
    event1_end = points[1]
    event2_start = points[2]
    event2_end = points[3]
    document = hash(text)

    if document in cache:
        recognized_events, recognized_relations = cache[document]
    else:
        recognized_events = extract_events(text)
        recognized_relations = recognize_relations(text, recognized_events, gold_graphs_precomputed[document])
        cache[document] = recognized_events, recognized_relations
        # print(document)
        for relation in recognized_relations:
            additional_relations[document].add(relation)

    events_all.add((event1_start, event1_end, document))
    events_all.add((event2_start, event2_end, document))
    e1_recognized = None
    e2_recognized = None
    used_recognized_events = set()
    for re in recognized_events:
        if re not in used_recognized_events and re[0] >= event1_start and re[0] <= event1_end or re[1] >= event1_start and re[1] <= event1_end or event1_start >= re[0] and event1_start <= re[1]:
            e1_recognized = re
            events_found.add(re)
            used_recognized_events.add(re)
        if re not in used_recognized_events and re[0] >= event2_start and re[0] <= event2_end or re[1] >= event2_start and re[1] <= event2_end or event2_start >= re[0] and event2_start <= re[1]:
            e2_recognized = re
            events_found.add(re)
            used_recognized_events.add(re)

    relation_recognized = False
    for rr in recognized_relations:
        if (rr[0] == e1_recognized and rr[1] == e2_recognized) or (rr[1] == e1_recognized and rr[0] == e2_recognized):
            if rr in additional_relations:
                additional_relations[document].remove(rr)
            relation_recognized = True

    if document not in documents:
        documents.add(document)
        document_texts[document] = text
        gold_graphs[document] = []
        predicted_graphs[document] = []

    # gold_graphs[document].append(correct)
    gold_graphs[document].append(((event1_start, event1_end, correct[0][2]), correct[1], (event2_start, event2_end, correct[2][2])))
    if e1_recognized is not None and e2_recognized is not None and relation_recognized:
        predicted_graphs[document].append(((event1_start, event1_end, predicted[0][2]), predicted[1], (event2_start, event2_end, predicted[2][2])))

print("events recall", len(events_found) / len(events_all))

events recall 0.8964269561284487


# Compute accuracy for teoretical event extractor

In [19]:
predictions

[((39, 48, 'dizziness'), 'OVERLAP', (283, 292, 'dizziness')),
 ((144, 153, 'described'), 'OVERLAP', (174, 194, 'unsteady on her feet')),
 ((128, 137, 'dizziness'), 'OVERLAP', (186, 206, 'unsteady on her feet')),
 ((116, 122, 'stated'),
  'BEFORE',
  (145, 197, 'trouble knowing if her feet were touching the ground')),
 ((106, 126, 'unsteady on her feet'),
  'OVERLAP',
  (162, 214, 'trouble knowing if her feet were touching the ground')),
 ((135, 144, 'dizziness'), 'OVERLAP', (186, 203, 'decreased hearing')),
 ((131, 148, 'decreased hearing'),
  'OVERLAP',
  (179, 197, 'static in her ears')),
 ((150, 168, 'static in her ears'),
  'OVERLAP',
  (180, 198, "meniere 's disease")),
 ((125, 132, 'head CT'),
  'BEFORE',
  (151, 196, 'a 1.5 cm diameter hyperdense mass in the pons')),
 ((100, 117, 'decreased hearing'),
  'OVERLAP',
  (219, 243, 'an oral prednisone taper')),
 ((88, 106, 'This unsteady gait'),
  'BEFORE',
  (174,
   240,
   'a pins and needle feeling in the face , hands and feet bi

In [20]:
correct_graph

[((39, 48, 'dizziness'), 'OVERLAP', (283, 292, 'dizziness')),
 ((144, 153, 'described'), 'BEFORE', (174, 194, 'unsteady on her feet')),
 ((128, 137, 'dizziness'), 'OVERLAP', (186, 206, 'unsteady on her feet')),
 ((116, 122, 'stated'),
  'BEFORE',
  (145, 197, 'trouble knowing if her feet were touching the ground')),
 ((106, 126, 'unsteady on her feet'),
  'OVERLAP',
  (162, 214, 'trouble knowing if her feet were touching the ground')),
 ((135, 144, 'dizziness'), 'BEFORE', (186, 203, 'decreased hearing')),
 ((131, 148, 'decreased hearing'),
  'OVERLAP',
  (179, 197, 'static in her ears')),
 ((150, 168, 'static in her ears'),
  'OVERLAP',
  (180, 198, "meniere 's disease")),
 ((125, 132, 'head CT'),
  'BEFORE',
  (151, 196, 'a 1.5 cm diameter hyperdense mass in the pons')),
 ((100, 117, 'decreased hearing'),
  'BEFORE',
  (219, 243, 'an oral prednisone taper')),
 ((88, 106, 'This unsteady gait'),
  'BEFORE',
  (174,
   240,
   'a pins and needle feeling in the face , hands and feet bilat

In [19]:
import random
# Generate graph
print("removed", "precision", "recall", "f1")
for remove_percentage in range(0, 100, 5):
    number_of_removed = int(remove_percentage * len(events) / 100)
    removed_events = random.sample(events, number_of_removed)
    correct_predictions = 0
    all_predictions = 0
    all_correct_predictions = 0
    
    predicted_size = 0
    gold_size = 0
    
    for predicted, correct in zip(predictions, correct_graph):
        event1 = predicted[0]
        event2 = predicted[2]
        if event1 in removed_events or event2 in removed_events:
            gold_size += 1
        else:
            gold_size += 1
            predicted_size += 1
            if predicted[1] == correct[1]:
                correct_predictions += 1
    p = correct_predictions / predicted_size
    r = correct_predictions / gold_size
    f1 = (2*p*r)/(p+r)
    print(remove_percentage, end=" ")
    print(p, end=" ")
    print(r, end=" ")
    print(f1)

removed precision recall f1
0 0.7294142172190953 0.7294142172190953 0.7294142172190954
5 0.7307736653983627 0.6606212215968313 0.6939289429025017
10 0.730563875504623 0.5847404627892433 0.649568691020668
15 0.737303985038124 0.5341880341880342 0.6195225143547899
20 0.7278877887788779 0.4597665207421305 0.5635620288744091
25 0.7263940520446097 0.4073379195330415 0.5219714171230132
30 0.733921815889029 0.36397748592870544 0.4866220735785953
35 0.733868943007783 0.30466958515739 0.43058112985195546
40 0.7322766570605187 0.26485303314571607 0.3890079608083282
45 0.7361530715005036 0.22858036272670418 0.3488427582915772
50 0.711139347734011 0.17500521159057744 0.2808866583019657
55 0.7347145022738757 0.15155305399207838 0.25127451827529595


KeyboardInterrupt: 

# compute F1 score

In [35]:
def overlap(event1, event2):
    start1, end1, text1 = event1
    start2, end2, text2 = event2
    if start1 >= start2 and start1 <= end2:
        # print(event1, event2)
        return True
    if end1 >= start2 and end1 <= end2:
        # print(event1, event2)
        return True
    if start2 >= start1 and start2 <= end1:
        # print(event1, event2)
        return True
    return False

def most_simmilar_event(event_list, event_target):
    # min_difference = -1
    # best_match = None
    for e in event_list:
        if overlap(e, event_target):
            return e
        # dif = -1
        # if e in event_target:
        #     dif = len(event_target) - len(e)
        # if event_target in e:
        #     dif = len(e) - len(event_target)
        # if dif >= 0 and (min_difference < 0 or min_difference > dif):
        #     min_difference = dif
        #     best_match = e
    return None

def find_relation(graph, event1, event2, expected_relation):
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2 and r[1] == expected_relation:
            return ind, r
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2:
            return ind, r
    return -1, None

# compare_graphs(truth, prediction)
def compare_graphs(graph1, graph2):
    event_map = {}
    events1 = list(set([r[0] for r in graph1]+[r[2] for r in graph1]))
    events2 = list(set([r[0] for r in graph2]+[r[2] for r in graph2]))
    for e in events1:
        event_map[e] = most_simmilar_event(events2, e)
    #print(event_map)
    
    matching_relation = [False for _ in range(len(graph2))]
    
    correct = 0
    incorrect = 0
    missing = 0
    too_much = 0
    for relation in graph1:
        ind2, relation2 = find_relation(graph2, event_map[relation[0]], event_map[relation[2]], relation[1])
        if relation2 is None:
            missing += 1
        else:
            matching_relation[ind2] = True
            if relation[1] == relation2[1]:
                correct += 1
            else:
                incorrect += 1
    too_much = len(matching_relation) - sum(matching_relation)
    #print(correct, incorrect, missing, too_much)
    return correct, incorrect, missing, too_much

def graph_intersect_size(graph1, graph2):
    graph1 = [a for a in graph1]
    graph2 = [a for a in graph2]
    intersect = []
    event_map = {}
    events1 = list(set([r[0] for r in graph1]+[r[2] for r in graph1]))
    events2 = list(set([r[0] for r in graph2]+[r[2] for r in graph2]))
    for e in events1:
        event_map[e] = most_simmilar_event(events2, e)
    #print(event_map)
    
    matching_relation = [False for _ in range(len(graph2))]
    
    correct = 0
    incorrect = 0
    missing = 0
    too_much = 0
    for relation in graph1:
        ind2, relation2 = find_relation(graph2, event_map[relation[0]], event_map[relation[2]], relation[1])
        if ind2 >= 0:
            intersect.append(relation2)
            graph2.pop(ind2)
    return len(intersect)

def compute_metrics_corrected(gold_graph, predicted_graph):
    gold_graph_index = 0
    correct_predictions = 0
    for predicted in predicted_graph:
        while not gold_graph[gold_graph_index][0] == predicted[0] or not gold_graph[gold_graph_index][2] == predicted[2]:
            gold_graph_index += 1
        # print(predicted[1], gold_graph[gold_graph_index][1])
        correct = predicted[1] == gold_graph[gold_graph_index][1]
        if correct:
            correct_predictions += 1
    precision = correct_predictions / len(predicted_graph)
    recall = correct_predictions / len(gold_graph)
    f1 = 2*(precision*recall)/(precision+recall)
    # print(f1)
    return correct_predictions, len(predicted_graph), len(gold_graph), precision, recall, f1

def find_relation(relations, event1, event2):
    graph = defaultdict(dict)
    for e1, rel, e2 in relations:
        graph[e1][e2] = rel
        graph[e2][e1] = inverse[rel]
    def dfs(current, target, visited):
        if current == target:
            return "OVERLAP"  # By default, same events are overlapping
        visited.add(current)
        for neighbor, rel in graph[current].items():
            if neighbor in visited:
                continue
            sub_rel = dfs(neighbor, target, visited)
            if sub_rel:
                return transitivity.get((rel, sub_rel), None)
        return None
    return dfs(event1, event2, set())

classes = ["BEFORE", "AFTER", "OVERLAP", "NONE"]
def compute_confusion_matrix(gold_graph, predicted_graph, confusion_matrix, additional_relations):
    gold_graph_index = -1
    correct_predictions = 0
    for predicted in predicted_graph:
        gold_graph_index += 1
        while not gold_graph[gold_graph_index][0] == predicted[0] or not gold_graph[gold_graph_index][2] == predicted[2]:
            # add missed relations
            confusion_matrix[len(classes) - 1][classes.index(gold_graph[gold_graph_index][1])] += 1
            gold_graph_index += 1
        # print(predicted[1], gold_graph[gold_graph_index][1])
        pred = predicted[1]
        gold = gold_graph[gold_graph_index][1]
        pred_index = classes.index(pred)
        gold_index = classes.index(gold)
        confusion_matrix[pred_index][gold_index] += 1
    if eval_scenario == "include_fp_against_gold":
        for additional_relation in additional_relations:
            if find_relation(gold_graph, additional_relation[0], additional_relation[1]) is None:
                confusion_matrix[0][len(classes) - 1] += 1


In [36]:
transitivity = {
    ("OVERLAP", "OVERLAP"): "OVERLAP",
    ("BEFORE", "OVERLAP"): "BEFORE",
    ("BEFORE", "BEFORE"): "BEFORE",
    ("OVERLAP", "BEFORE"): "BEFORE",
    ("AFTER", "OVERLAP"): "AFTER",
    ("OVERLAP", "AFTER"): "AFTER",
    ("AFTER", "AFTER"): "AFTER",
    ("AFTER", "BEFORE"): "OVERLAP",
    ("BEFORE", "AFTER"): "OVERLAP"
}

inverse = {
    "BEFORE": "AFTER",
    "AFTER": "BEFORE",
    "OVERLAP": "OVERLAP"
}

In [37]:
def does_relation_exist(graph, event1, event2):
    ind, relation = find_relation(graph, event1, event2, None)
    return ind >= 0

def add_transitive(graph):
    for i in range(len(graph)):
        for j in range(i+1, len(graph)):
            if graph[i][2] == graph[j][0]:
                # matching relations
                event1 = graph[i][0]
                event2 = graph[j][2]
                if not does_relation_exist(graph, event1, event2):
                    relation = transitivity[(graph[i][1], graph[j][1])]
                    new_relation = (event1, relation, event2)
                    graph.append(new_relation)
    return graph

def add_inverse(graph):
    for i in range(len(graph)):
        event1 = graph[i][2]
        event2 = graph[i][0]
        if not does_relation_exist(graph, event1, event2):
            relation = inverse[graph[i][1]]
            new_relation = (event1, relation, event2)
            graph.append(new_relation)
    return graph

def graph_closure(graph):
    graph = add_inverse(graph)
    graph = add_transitive(graph)
    return graph

In [38]:
confusion_matrix = [[0 for _ in range(len(classes))] for _ in range(len(classes))]
for doc in documents:
    gold_graph = gold_graphs[doc]
    predicted_graph = predicted_graphs[doc]
    # print(predicted_graph)
    additional = additional_relations[doc]
    # print(additional)
    
    compute_confusion_matrix(gold_graph, predicted_graph, confusion_matrix, additional)

In [39]:
import numpy as np
cm = np.array(confusion_matrix)
true_pos = np.diag(cm)
false_pos = np.sum(cm, axis=1) - true_pos
false_neg = np.sum(cm, axis=0) - true_pos
print("confusion matrix")
for row in confusion_matrix:
    print(row)
print("results")
print(true_pos)
print(false_pos)
print(false_neg)

p = np.nan_to_num(true_pos / (true_pos+false_pos))
r = np.nan_to_num(true_pos / (true_pos+false_neg))
f1 = np.nan_to_num(2 * p * r / (p + r))
print("p:", p)
print("r:", r)
print("f1:", f1)
ma_p = np.sum(p * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
ma_r = np.sum(r * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
ma_f1 = np.sum(f1 * (true_pos+false_neg)) / np.sum((true_pos+false_neg))
print("precision", ma_p)
print("recall", ma_r)
print("f1", ma_f1)

confusion matrix
[1103, 134, 406, 0]
[190, 567, 268, 0]
[727, 496, 4531, 0]
[324, 169, 659, 0]
results
[1103  567 4531    0]
[ 540  458 1223 1152]
[1241  799 1333    0]
p: [0.67133293 0.55317073 0.78745221 0.        ]
r: [0.47056314 0.41508053 0.77268076 0.        ]
f1: [0.55329822 0.47427854 0.77999656 0.        ]
precision 0.7255959206734337
recall 0.6476916649258408
f1 0.6808747992810781


# Error analysis

In [ ]:
doc = list(documents)[-1]
gold_graph = gold_graphs[doc]
pred_graph = predicted_graphs[doc]

gold_graph = sorted(gold_graph, cmp=lambda item1, item2: item1[0][0] - item2[0][0])
pred_graph = sorted(pred_graph, cmp=lambda item1, item2: item1[0][0] - item2[0][0])
print(document_texts[doc])

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
def visualize_graph(graph):
    events = set()
    edges = {}
    for triplet in graph:
        events.add(triplet[0][2])
        events.add(triplet[2][2])
        edges[(triplet[0][2], triplet[2][2])] = triplet[1]
    events = list(events)
    G = nx.Graph()
    G.add_nodes_from(events)
    G.add_edges_from(edges.keys())

    plt.figure()
    pos = nx.spring_layout(G)
    nx.draw(
        G, pos, edge_color='black', width=1, linewidths=1,
        node_size=100, node_color='pink', alpha=0.9,
        labels={node: node for node in G.nodes()}
    )
    # nx.draw_networkx_edges(
    #     G,
    #     pos,
    #     node_size=100,
    #     arrowstyle="->",
    #     arrowsize=10,
    #     width=2,
    # )
    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=edges,
        font_color='red'
    )
    plt.axis('off')
    plt.show()
    
visualize_graph(pred_graph[3:7])

In [ ]:
# can visualise at https://csacademy.com/app/graph_editor/
def convert_graph_format(graph):
    events = set()
    edges = {}
    for triplet in graph:
        e1 = triplet[0][2].replace(" ", "-")
        e2 = triplet[2][2].replace(" ", "-")
        events.add(e1)
        events.add(e2)
        edges[(e1, e2)] = triplet[1]
    events = list(events)

    print(len(events))
    for e in events:
        print(e)
    for edge in edges:
        print(edge[0], edge[1], edges[edge])

In [ ]:
visualize_graph(pred_graph)
convert_graph_format(pred_graph)

In [ ]:
visualize_graph(gold_graph)
convert_graph_format(gold_graph)